In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
BRONZE_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/bronze"
SILVER_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/silver"
GOLD_PATH = PROJECT_ROOT / "data/gulf_stream_pigment_influencers_20241001_20251231_v1/gold"

pd.set_option("display.max_rows", 1000)

### Bailey and Werdell (2006) validation procedure

For each satellite and in situ pair:

- Use native-resolution satellite data instead of reduced-resolution data.
- Require the in situ measurement to be within 3 hours of the satellite overpass.
- Use a 5 x 5 pixel box centered on the in situ location.
- Require each satellite record to be unique and to share no pixels with another validation record.
- Exclude data with a sensor zenith angle greater than 60 degrees.
- Exclude data with a solar zenith angle greater than 75 degrees.
- Mask pixels affected by land, cloud or ice, stray light, high sun glint, saturation, low water-leaving radiance at 555 nm, or atmospheric-correction failure.
- Apply additional flags only when they are specific to the product under validation.
- Require at least 50% of the 25 pixels to remain valid.
- For coastal water, require at least 50% of non-land pixels and at least 5 valid pixels.
- Remove values outside the initial mean plus or minus 1.5 standard deviations.
- Compute the coefficient of variation from the filtered mean and standard deviation.
- Reject the box when the median coefficient of variation exceeds 0.15.
- Exclude optically shallow records when physical depth is less than `1.3 / K490`.

The coefficient of variation test uses water-leaving radiance from 412 through 555 nm and aerosol optical thickness at 865 nm.

Source: [Bailey and Werdell (2006)](https://doi.org/10.1016/j.rse.2006.01.015).

In [ ]:
# Keep only the relevant cols

from eddy_tracking.validation.seabass import read_hplc_dir

df = read_hplc_dir("../data/pvst_bats_hplc")
df["datetime"] = df["datetime"].dt.tz_localize("UTC")

# depth in meters
# depth is sample depth below water surface
# water_depth is distance from water surface to sea floor
# Anything < 200 m is coastal water; > 1000 m is open ocean
meta_cols = ["depth", "water_depth", "datetime", "lon", "lat"]

# SeaBASS column -> SDP model pigment
seabass_to_sdp = {
    "allo": "Allo",
    "but-fuco": "ButFuco",
    "chl_c1c2": "chl c1+c2",
    "chl_c3": "chl c3",
    "dv_chl_a": "DV chla",
    "fuco": "Fuco",
    "hex-fuco": "HexFuco",
    "mv_chl_b": "MV chlb",
    "neo": "Neo",
    "perid": "Perid",
    "tot_chl_a": "T chla",
    "viola": "Viola",
    "zea": "Zea",
}
pigment_cols = list(seabass_to_sdp.keys())

keep_cols = set(meta_cols) | set(pigment_cols)
df = df.loc[:, df.columns.isin(keep_cols)]
df.head()

In [ ]:
# Download matchup files
# Read one or more matchups per unique (lon, lat, dttm)
# Apply QA flags -> remove rows that don't pass the filter
# Generate mapping of (lon, lat, dttm) -> pd.DataFrame of all relevant pixels within 3 hour period containing that point
# Limit only to the 5 geographically closest points in terms of lon/lat (becomes 5x5 box)
# This is slightly different from Bailey & Werdell but it's probably easier for us (first find closest pixel, then have a 5x5 native box using scan_line/pixel, then apply QA within box, then require at least 50% to be valid, then compute filtered satellite value)
# We remove disqualifying rows first and just take the filtered arithmetic mean

# PACE matchup -> 5x5 filter -> QA flags
# if 50%+ match then SDP -> filtered mean -> compare w/ in-situ HPLC

from datetime import timedelta

import earthaccess

from eddy_tracking.validation.matchup import list_pace_l2_matchups, list_sss_matchups, list_sst_matchups
from eddy_tracking.utils.authentication import login_earthdata
from eddy_tracking.preprocess.pace import read_multiple_pace_l2
from eddy_tracking.preprocess.sss import read_multiple_sss
from eddy_tracking.preprocess.sst import read_multiple_sst
from eddy_tracking.validation.quality import apply_l2_quality_flags
from eddy_tracking.utils.geography import get_5x5_pace_l2_matchups

def filtered_arithmetic_mean(values: pd.Series) -> float:
    values.dropna(inplace=True)
    if values.empty:
        return float("nan")

    mean = values.mean()
    std = values.std()
    filtered_values = values[(values > mean - 1.5 * std) & (values < mean + 1.5 * std)]
    return filtered_values.mean()

login_earthdata()
pace_l2_download_dir = Path("../data/sdp_validation/pace_l2")
sss_download_dir = Path("../data/sdp_validation/sss")
sst_download_dir = Path("../data/sdp_validation/sst")
pace_l2_download_dir.mkdir(parents=True, exist_ok=True)
sss_download_dir.mkdir(parents=True, exist_ok=True)
sst_download_dir.mkdir(parents=True, exist_ok=True)

unique_measurements = df[["lon", "lat", "datetime"]].drop_duplicates().reset_index(drop=True)
window = timedelta(hours=3)

matchups = {} # (lon, lat, dttm) -> QA-filtered pd.DataFrame
downloaded_ct = 0
for lon, lat, dttm in unique_measurements.itertuples(index=False, name=None):
    # Download data
    pace_granules = list_pace_l2_matchups(lon, lat, dttm, window)
    downloaded_fps = earthaccess.download(
        granules=pace_granules,
        local_path=pace_l2_download_dir,
    )
    sss_granules = list_sss_matchups(lon, lat, dttm, window)
    earthaccess.download(
        granules=sss_granules,
        local_path=sss_download_dir,
    )
    sst_granules = list_sst_matchups(lon, lat, dttm, window)
    earthaccess.download(
        granules=sst_granules,
        local_path=sst_download_dir,
    )

    sss_matchup_df = read_multiple_sss(sss_download_dir)
    sst_matchup_df = read_multiple_sst(sst_download_dir)
    pace_matchup_df = read_multiple_pace_l2(downloaded_fps)

    
    filtered_matchup_df = apply_l2_quality_flags(pace_matchup_df)
    filtered_matchup_df = get_5x5_pace_l2_matchups(filtered_matchup_df, lon, lat)

    

print(f"{downloaded_ct} downloaded")